# 《PythAPCS123》單元 13-2：語法錯誤（SyntaxError）深度排查與 Traceback 閱讀心法

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-2_syntax_errors_and_traceback_decoding.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：克服初學者面對直譯器紅色英文報錯（Traceback）的內心恐懼，徹底搞懂「錯誤行號（Line Number）」與「游標指標（^）」的真實座標意義，地毯式拆解考場最常見的漏冒號、括號引號不對稱、縮排空格混用、賦值等號誤用及全形標點符號等五大語法地雷，練就 10 秒精準鎖定並修復 SyntaxError 的必備除錯神技。


### 13.2.1 直譯器編譯檢查期：為什麼程式碼尚未執行就先報錯？

許多初學者在剛開始學習 Python 時，常常存在一種根深蒂固的直覺誤解：「Python 是直譯語言，所以它一定是一行一行讀取、讀到哪一行就執行哪一行；因此，只要我的程式前面幾行沒寫錯，電腦至少應該先把前幾行的 `print()` 印出來，等跑到出錯的那一行才會當機停下來吧？」

當初學者滿懷期待地按下執行鍵，卻發現畫面上「連半個字都沒印出來」，直譯器就立刻在第 10 行噴出紅色的 `SyntaxError` 時，往往會感到無比困惑。這是因為 Python 在執行任何程式碼之前，內部必須先經歷一段極其迅速的**「語法剖析與位元組編譯期（Parsing & Bytecode Compilation Phase）」**。直譯器會先把整份原始程式碼（.py 檔案）當作一篇完整的文章從頭到尾掃描一遍，並將人類看得懂的程式文字轉換為電腦底層能夠高效處理的「位元組碼（Bytecode）」。

只要整篇文章中有任何一個單字拼錯、括號沒關、縮排錯位或漏掉冒號，語法剖析器（Parser）就會認定整份文法結構不合規格，直譯器會當機立斷「拒絕啟動虛擬機（Python Virtual Machine, PVM）」，連第 1 行的 `print("程式開始")` 都不會執行！理解這個靜態檢查階段至關重要：語法錯誤是直譯器在門口就把你攔截下來，它與程式執行到一半崩潰的「執行時期錯誤（Runtime Error）」有著本質上的不同。


In [ ]:
# 13.2.1 程式碼演示：證明 SyntaxError 會阻止整份程式碼啟動執行
import sys

demo_script = '''
print("=== 步驟 1: 程式準備啟動 ===")
print("=== 步驟 2: 正在處理前置作業 ===")

# 故意在第 6 行埋下致命語法錯誤：if 漏掉冒號
if 10 > 5
    print("這行永遠不會被看到")
'''

print("--- 模擬執行包含 SyntaxError 的腳本 ---")
try:
    # 嘗試編譯並執行字串代碼
    exec(demo_script)
except SyntaxError as e:
    print(f"\n[直譯器回報] 捕捉到語法錯誤 SyntaxError！")
    print(f"錯誤行號: 第 {e.lineno} 行")
    print(f"錯誤描述: {e.msg}")
    print("觀察重點：畫面中完全沒有出現『步驟 1』或『步驟 2』的輸出！")
    print("結論：語法錯誤發生在編譯期，整份程式連一個字元都不會被執行。")


### 13.2.1 語法重點回顧與核心觀念提煉

在剛才的程式碼驗證中，我們親眼見證了：即使第 1 行與第 2 行的 `print()` 完全合法且人畜無害，但因為第 6 行存在 `if 10 > 5` 漏掉冒號的語法硬傷，整個腳本在編譯期就被直譯器全面封鎖，根本無法進入執行階段。

這為我們提供了兩大極具價值的除錯心智模型：
1. **絕不要以為「前面有對就會先跑」**：如果你的程式碼上傳到 OJ 後直接拿到 CE，代表整份程式在編譯期就被攔截，連第一行讀入測資的 `input()` 都沒摸到。
2. **語法錯誤是最高優先級障礙**：在思考演算法邏輯、資料結構或效能優化之前，必須先保證語法結構完全符合 Python 文法規則。只要有語法錯誤存在，任何邏輯測試都是空談。


In [ ]:
# 13.2.1 學生實作練習：語法編譯檢查器
# 任務說明：實作 is_syntax_valid(code_str) 函式
# 若傳入的程式碼字串符合 Python 語法結構（能順利通過 compile），回傳 True
# 若存在語法錯誤拋出 SyntaxError，回傳 False

def is_syntax_valid(code_str: str) -> bool:
    # 請在此處使用 try-except 與 compile() 進行語法檢查
    pass

# 測試用例
print("測試 1 (合法):", is_syntax_valid("a = 10\nb = 20\nprint(a + b)"))
print("測試 2 (不合法):", is_syntax_valid("for i in range(5)\n    print(i)"))


In [ ]:
# 13.2.1 單元測試驗證
def is_syntax_valid_ans(code_str: str) -> bool:
    try:
        compile(code_str, filename="<test>", mode="exec")
        return True
    except SyntaxError:
        return False

assert is_syntax_valid_ans("x = 5\nprint(x)") == True
assert is_syntax_valid_ans("if x == 5\n    print(x)") == False
assert is_syntax_valid_ans("a = [1, 2, 3") == False
assert is_syntax_valid_ans("def foo():\n    return 42") == True

print("🎉 13.2.1 所有測試通過！成功領悟直譯器語法編譯期機制！")


### 13.2.2 看懂 Traceback 座標軸：錯誤行號（Line number）與游標指標 `^` 定位

每當終端機噴出一整片紅色的英文錯誤追蹤訊息（Traceback）時，許多零基礎學生的第一反應往往是「眼神迴避、心跳加速、不知所措」，甚至下意識立刻把整個視窗關掉重開。請記住：**Traceback 不是電腦在罵你，而是電腦在向你發出最精準的求救訊號！**

在所有語法錯誤的 Traceback 訊息中，直譯器提供了一套非常精準的「二維空間座標定位系統」：
1. **第一維度：檔案名稱與行號（`File "...", line X`）**：這是直譯器的 Y 軸座標。它明確告訴你，解析器是在第幾行撞牆的。
2. **第二維度：原始程式碼片段（Code Text）**：直譯器會把出問題的那一行程式碼完整印出來給你看。
3. **第三維度：游標箭頭指標（Caret Symbol `^` 或波浪線 `~~~`）**：這是直譯器的 X 軸座標。在 Python 3.10+ 版本中，強化版直譯器會用一個或多個 `^` 與 `~`，極其精準地指著它「看到哪一個字元發現不合法」。
4. **第四維度：錯誤類型與原因描述（Error Message）**：在最底下一行，如 `SyntaxError: expected ':'`（預期需要冒號）或 `SyntaxError: unmatched ')'`（不匹配的多餘右括號）。只要冷靜看懂這四個維度，除錯就只是按圖索驥的簡單操作！


In [ ]:
# 13.2.2 程式碼演示：解析現代 Python 3.10+ 游標指標 (Caret ^) 的精確座標定位
import traceback

def parse_and_display_syntax_error(code_text):
    '''
    示範如何從 SyntaxError 物件中提取出精準的行列座標與指標符號
    '''
    print("=== 正在解析目標程式碼 ===")
    try:
        compile(code_text, "<eval_code>", "exec")
        print("  [PASS] 語法無誤！\n")
    except SyntaxError as e:
        print("  [Traceback 解析結果]")
        print(f"  --> 出錯檔案: {e.filename}")
        print(f"  --> 出錯行號: 第 {e.lineno} 行 (Y軸)")
        print(f"  --> 出錯欄位: 第 {e.offset} 個字元 (X軸)")
        print(f"  --> 問題代碼: {e.text.strip() if e.text else 'N/A'}")
        
        # 繪製精準的游標箭頭
        indent = " " * (e.offset - 1) if e.offset else ""
        print(f"  --> 游標箭頭: {indent}^ (直譯器在此處撞牆！)")
        print(f"  --> 錯誤診斷: {e.msg}\n")

# 案例 A: 算術表達式中的多餘符號
parse_and_display_syntax_error("total = 100 + * 5")

# 案例 B: 漏掉右括號
parse_and_display_syntax_error("nums = [1, 2, 3\nprint('done')")


### 13.2.2 語法重點回顧與核心觀念提煉

在剛才的解析工具中，我們清楚看到了 Python 如何透過 `e.lineno` 與 `e.offset` 標記出立體座標。然而，這裡有一個非常隱蔽的「考場盲點」需要特別警戒：

**「上行漏括號，下行遭池魚之殃」的移位陷阱**：
請仔細觀察剛才的案例 B！出錯的根本原因是第 1 行的串列 `nums = [1, 2, 3` 漏掉了右方括號 `]`。但是，直譯器報錯的行號卻是**第 2 行**的 `print('done')`！
為什麼會這樣？因為在 Python 文法中，一個左括號或左方括號開啟後，允許跨行書寫元素。直譯器在讀完第 1 行後，以為第 2 行的文字還是方括號內部的內容，直到看見 `print` 指令時才發現文法崩潰！

**黃金除錯心法**：每當直譯器報錯指出某一行有語法錯誤，但你左看右看那一行明明寫得完全正確時，**請立刻把視線往上移一行（Line - 1）**！99% 的機率是上一行的括號、引號沒有閉合！


In [ ]:
# 13.2.2 學生實作練習：定位並修復移位括號語法錯誤
# 題目說明：以下程式碼在執行時，直譯器回報第 4 行出錯
# 請找出真正的錯誤位置（其實在第 3 行！），修正程式碼字串並確認可順利編譯。

buggy_multiline_code = '''
def process_data():
    items = [10, 20, 30
    total = sum(items)
    return total
'''

# 請將修正後的程式碼填入 fixed_multiline_code 中
fixed_multiline_code = '''
# 請修正後填入
'''

# 驗證
parse_and_display_syntax_error(fixed_multiline_code)


In [ ]:
# 13.2.2 單元測試驗證
sample_fixed = '''
def process_data():
    items = [10, 20, 30]
    total = sum(items)
    return total
'''

comp_success = False
try:
    compile(sample_fixed, "<test>", "exec")
    comp_success = True
except SyntaxError:
    comp_success = False

assert comp_success, "修復後的代碼必須能通過編譯！"
print("🎉 13.2.2 所有測試通過！成功建立 Traceback 座標判讀與上行移位排查眼力！")


### 13.2.3 考場高頻語法雷區一：漏冒號 `:` 與括號/引號不對稱

根據歷年 APCS 實作測驗與程式新手統計，有超過 60% 的語法錯誤都是由兩個最簡單的標點符號引起的：**冒號（Colon `:`）** 與 **成對符號（Brackets & Quotes）**。

#### 雷區一：條件與迴圈結尾「漏冒號（Missing Colon）」
在 Python 中，冒號 `:` 是引導「縮排程式區塊（Code Block）」的唯一鑰匙。舉凡 `if`、`elif`、`else`、`for`、`while`、`def`、`class`、`try`、`except`，每一條複合語句（Compound Statement）的行尾都必須緊跟著一個英文半形冒號 `:`。
- 錯誤範例：`if score >= 60` ➔ 報錯：`SyntaxError: expected ':'`
- 心理成因：學生在紙上寫數學式習慣了，手腦脫節漏敲鍵盤。

#### 雷區二：引號字串「開頭有、結尾無（Unterminated String Literal）」
字串必須以成對的單引號 `'...'` 或雙引號 `"..."` 包裹，若開頭用單引號、結尾用雙引號，或是行尾漏打引號，直譯器會立刻拋出 `SyntaxError: unterminated string literal`。
- 錯誤範例：`msg = "Hello World'` ➔ 引號混用不匹配！
- 錯誤範例：`msg = "Hello World` ➔ 漏打右引號！

在接下來的實作中，我們將練習專門針對這兩大高頻雷區進行快速肌肉記憶掃描。


In [ ]:
# 13.2.3 程式碼演示：漏冒號與引號不對稱的直譯器報錯特徵
def test_syntax_snippet(snippet_name, code_str):
    print(f"--- 測試案例: {snippet_name} ---")
    try:
        compile(code_str, "<snippet>", "exec")
        print("  [AC] 語法結構完全正確！\n")
    except SyntaxError as e:
        print(f"  [CE 捕捉] 第 {e.lineno} 行: {e.msg}")
        print(f"  代碼片段: {repr(code_str)}")
        print()

# 案例 1: 函式宣告漏冒號
test_syntax_snippet("函式 def 漏冒號", "def add(a, b)\n    return a + b")

# 案例 2: 迴圈 for 漏冒號
test_syntax_snippet("迴圈 for 漏冒號", "for i in range(5)\n    print(i)")

# 案例 3: 引號不對稱 (雙引號開頭，單引號結尾)
test_syntax_snippet("字串引號混用", 'name = "Alice\'')

# 案例 4: 修正後的完美代碼
test_syntax_snippet("修正後的正常程式", "def add(a, b):\n    return a + b")


### 13.2.3 語法重點回顧與核心觀念提煉

從剛才的報錯訊息中，我們可以歸納出直譯器對這兩類錯誤的標準關鍵字：
1. `expected ':'`：這是直譯器最直白的喊話——「我在這行末尾等你的冒號，你竟然忘了給我！」
2. `unterminated string literal`：代表「未終止的字串常數」，只要看見 `unterminated` 這個單字，不用懷疑，立刻檢查該行的引號是不是漏關閉或單雙引號混搭了。

**防禦反射神經建立**：
在編寫 Python 程式碼時，請培養「成對輸入」的肌肉記憶：在鍵入引號時連按兩次 `""` 或 `''`，再將游標移入中間輸入文字；打完 `if` / `for` / `def` 的條件後，右手小指立刻反射性敲擊冒號鍵 `:`。這套微小習慣能幫你在 APCS 考場上省下大把無謂的除錯時間。


In [ ]:
# 13.2.3 學生實作練習：自動修補冒號與引號工具雛形
# 任務說明：實作 check_control_colon(line)
# 給定單行 Python 控制語句字串（已去除首尾空白）
# 若該行以 "if ", "for ", "while ", "def ", "else", "elif " 開頭，且結尾「不是冒號 :」
# 則自動在結尾補上冒號 ":" 並回傳；若已有冒號或不是控制語句，則原樣回傳。

def check_control_colon(line: str) -> str:
    # 請在此處撰寫補冒號邏輯
    pass

# 測試用例
print("測試 1:", check_control_colon("if score >= 60"))   # 應回傳 "if score >= 60:"
print("測試 2:", check_control_colon("for x in arr:"))     # 應回傳 "for x in arr:" (已有)
print("測試 3:", check_control_colon("ans = a + b"))       # 應回傳 "ans = a + b" (非控制語句)


In [ ]:
# 13.2.3 單元測試驗證
def check_control_colon_ans(line: str) -> str:
    s = line.strip()
    keywords = ("if ", "for ", "while ", "def ", "else", "elif ")
    if any(s.startswith(kw) for kw in keywords) or s in ("else:", "else"):
        if not s.endswith(":"):
            return s + ":"
    return s

assert check_control_colon_ans("if score >= 60") == "if score >= 60:"
assert check_control_colon_ans("while count > 0") == "while count > 0:"
assert check_control_colon_ans("def solve()") == "def solve():"
assert check_control_colon_ans("else") == "else:"
assert check_control_colon_ans("for i in range(5):") == "for i in range(5):"
assert check_control_colon_ans("x = 10") == "x = 10"

print("🎉 13.2.3 所有測試通過！成功掌握冒號與引號雷區防禦機制！")


### 13.2.4 考場高頻語法雷區二：縮排不一致 `IndentationError` 與全形/半形空白混用災難

在 C++ 或 Java 中，程式區塊是由大括號 `{ ... }` 界定的；但在 Python 中，**「縮排（Indentation）就是語法本身」**！這是 Python 最優雅的特色，但也是初學者最容易踩雷的深水區。

由縮排引發的語法硬傷主要有三種形態：
1. **`IndentationError: expected an indented block`（預期縮排區塊）**：在 `if`、`for`、`def` 冒號的下一行，必須至少縮排一層。如果冒號後面直接接頂格程式碼，或是區塊內空空如也（忘了寫 `pass`），直譯器就會報出此錯。
2. **`IndentationError: unexpected indent`（突兀的非預期縮排）**：在平行的指令之間，某一行無緣無故多敲了一個空格，直譯器會困惑地抗議：「沒有人在引導你，你為什麼自己縮進去了？」
3. **`TabError: inconsistent use of tabs and spaces in indentation`（Tab 鍵與空白鍵混用大災難）**：這是肉眼最難察覺的幽靈 Bug！在文字編輯器裡，1 個 Tab 和 4 個空白鍵看起來「寬度一模一樣」，但對直譯器而言，Tab 是 ASCII `0x09`，空格是 ASCII `0x20`，本質完全不同！在 Python 3 中，嚴格禁止在同一個程式碼區塊內混用 Tab 與空白鍵。

更致命的是**「全形空白字元（Full-width Space, `\u3000`）」**：當學生切換為中文輸入法打註解後，不慎在行首敲下了全形空白鍵，肉眼看似正常留白，直譯器卻會直接引爆 `SyntaxError: invalid character in identifier`！


In [ ]:
# 13.2.4 程式碼演示：揭開隱形全形空白與 Tab/空格混用的真面目
def inspect_invisible_whitespace(code_string):
    '''
    將程式碼中的隱形字元具象化顯示：
    - 半形空格顯示為 '·'
    - Tab 鍵顯示為 '⇥'
    - 全形中文空白顯示為 '【全形空白!】'
    '''
    print("=== 隱形字元透視鏡檢驗 ===")
    lines = code_string.splitlines()
    for lineno, line in enumerate(lines, 1):
        visible_line = ""
        has_fullwidth = False
        for ch in line:
            if ch == ' ':
                visible_line += "·"
            elif ch == '\t':
                visible_line += "⇥   "
            elif ch == '\u3000':
                visible_line += "[全形空白!]"
                has_fullwidth = True
            else:
                visible_line += ch
        print(f"Line {lineno}: {visible_line}")
        if has_fullwidth:
            print(f"  --> 警告！第 {lineno} 行偵測到中文全形空白 \\u3000，將引發 SyntaxError！")
    print()

# 案例 1: 混入致命中文全形空白
bad_fullwidth_code = "def hello():\n　　print('hi')" # 這裡的縮排是中文全形空白！
inspect_invisible_whitespace(bad_fullwidth_code)

try:
    compile(bad_fullwidth_code, "<test>", "exec")
except SyntaxError as e:
    print(f"直譯器真實報錯: {type(e).__name__} - {e.msg}\n")

# 案例 2: 正常的標準 4 空格縮排
good_code = "def hello():\n    print('hi')"
inspect_invisible_whitespace(good_code)


### 13.2.4 語法重點回顧與核心觀念提煉

透過剛才的「隱形字元透視鏡」，原本潛伏在程式碼中的全形空白 `\u3000` 瞬間無所遁形。這種錯誤在初學者身上極為常見，特別是在切換中文輸入法寫作業註解、又切回寫程式碼的交界處，極易誤觸全形空白鍵。

**縮排避坑三大黃金守則**：
1. **嚴格統一使用 4 個半形空格（Space）**：在所有現代編輯器（VS Code, Colab, Jupyter）中，將 Tab 鍵行為設定為「自動插入 4 個空格（Insert Spaces）」，永遠不向檔案寫入真正的 Tab 字元。
2. **輸入法隨時切換純英文（EN / 半形）**：寫程式碼符號與縮排時，確認右下角為半形英數狀態，杜絕中文全形符號與全形空白的污染。
3. **冒號後若暫無實作，務必補上 `pass`**：若先寫了 `if condition:` 框架而尚未構思細節，下一行請縮排寫上 `pass` 充當占位符，防止直譯器抱怨 `expected an indented block`。


In [ ]:
# 13.2.4 學生實作練習：全形空白淨化器
# 任務說明：實作 sanitize_indentation(code_str)
# 將字串中所有的中文全形空白 '\u3000' 替換為兩個半形空格 '  '
# 並將所有 Tab 字元 '\t' 替換為四個半形空格 '    '
# 回傳清洗乾淨的純淨 Python 程式碼

def sanitize_indentation(code_str: str) -> str:
    # 請在此處撰寫字元替換與清洗邏輯
    pass

# 測試用例
polluted = "def test():\n　　\tprint('clean me')"
cleaned = sanitize_indentation(polluted)
print("清洗前透視：")
inspect_invisible_whitespace(polluted)
print("清洗後透視：")
inspect_invisible_whitespace(cleaned)


In [ ]:
# 13.2.4 單元測試驗證
def sanitize_indentation_ans(code_str: str) -> str:
    res = code_str.replace('\u3000', '  ')
    res = res.replace('\t', '    ')
    return res

test_input = "def foo():\n　　x = 10\n\tprint(x)"
output = sanitize_indentation_ans(test_input)
assert '\u3000' not in output, "字串中不得再殘留全形空白！"
assert '\t' not in output, "字串中不得再殘留 Tab 字元！"
assert "    print(x)" in output, "Tab 應被替換為 4 個半形空格！"

print("🎉 13.2.4 所有測試通過！徹底征服縮排與全形空白隱形地雷！")


### 13.2.5 考場高頻語法雷區三：等號賦值 `=` 與比較 `==` 誤用，及中文標點混淆

在初學者的腦袋裡，「單等號 `=`」與「雙等號 `==`」是兩具常常打架的雙胞胎：
- **單等號 `=` 是賦值（Assignment）**：代表「將右邊的值裝入左邊的變數箱子中」。等號左邊必須是合法的變數名稱，不能是常數或算式！
- **雙等號 `==` 是比較（Equality Comparison）**：代表「詢問兩邊的值是否相等」，回傳布林值 `True` 或 `False`。

#### 致命失誤一：在 `if` 條件式中誤用單等號賦值
在 C/C++ 中，`if (x = 5)` 不會報錯，但會引發嚴重的邏輯災難；而在 Python 中，語法剖析器更加嚴謹，直接將其列為**語法硬傷**！
- 錯誤範例：`if x = 10:` ➔ 直譯器直接拒絕啟動，回報 `SyntaxError: invalid syntax. Maybe you meant '==' or ':='?`！

#### 致命失誤二：將值賦給算式或常數
- 錯誤範例：`10 = x` 或 `a + b = 5` ➔ 報錯：`SyntaxError: cannot assign to expression` 或 `cannot assign to literal`！等號左邊絕不能是數字常數或算術表達式。

#### 致命失誤三：中文全形標點符號混淆
在輸入逗號、括號、引號時，若不慎使用中文輸入法打出全形符號（如中文逗號 `，`、全形雙引號 `“”`、全形冒號 `：`），直譯器無法將其辨識為語法運算子，會直接噴出 `SyntaxError: invalid character`！在下面的範例中，我們將實作一個標點符號照妖鏡。


In [ ]:
# 13.2.5 程式碼演示：等號誤用與全形標點符號報錯解析
def check_assignment_and_punctuation(code_str):
    print(f"--- 檢驗代碼: {repr(code_str)} ---")
    try:
        compile(code_str, "<check>", "exec")
        print("  [PASS] 語法合法！\n")
    except SyntaxError as e:
        print(f"  [CE 捕捉] {type(e).__name__}: {e.msg}")
        print(f"  問題位置: 行 {e.lineno}, 偏移 {e.offset}\n")

# 案例 1: if 條件誤用單等號賦值
check_assignment_and_punctuation("if score = 100:\n    print('Perfect')")

# 案例 2: 等號左邊是算術表達式
check_assignment_and_punctuation("a + b = 20")

# 案例 3: 混入全形中文逗號
check_assignment_and_punctuation("print('A'，'B')")  # 這裡的逗號是中文全形 ，


### 13.2.5 語法重點回顧與核心觀念提煉

剛才直譯器的貼心提示 `Maybe you meant '=='?` 清楚點出了初學者最常犯的心智混淆。在程式語言的世界裡，「賦值」與「相等判定」是壁壘分明的兩種動作：

1. **條件判斷一律用雙等號 `==`**：無論是 `if`、`elif` 還是 `while`，只要你的目的是「檢查是否相等」，請務必連敲兩次等號鍵 `==`。
2. **賦值運算左側必須為乾淨的「變數名稱」**：算式永遠只能寫在等號右側（`sum_val = a + b` 是正確的，`a + b = sum_val` 必定 CE）。
3. **辨識全形標點符號的視覺技巧**：
   - 半形逗號 `,` 後方通常緊貼底線，高度較低；全形逗號 `，` 佔據完整正方形字幅，位於正中央。
   - 半形括號 `()` 瘦長小巧；全形括號 `（）` 圓胖寬大。每當直譯器回報 `invalid character` 時，先按退格鍵將該處符號刪除，切換純半形英數重新鍵入一次。


In [ ]:
# 13.2.5 學生實作練習：中文全形標點符號一鍵轉換器
# 任務說明：實作 convert_fullwidth_symbols(text)
# 將字串中常見的中文全形標點符號替換為標準英文半形符號：
# '，' -> ','
# '：' -> ':'
# '（' -> '('
# '）' -> ')'
# '；' -> ';'

def convert_fullwidth_symbols(text: str) -> str:
    # 請在此處撰寫全形轉半形字典對應替換邏輯
    pass

# 測試用例
sample_code_with_chinese_symbols = "if (x == 10)：\n    print(a，b)；"
converted = convert_fullwidth_symbols(sample_code_with_chinese_symbols)
print("轉換前：\n", sample_code_with_chinese_symbols)
print("轉換後：\n", converted)


In [ ]:
# 13.2.5 單元測試驗證
def convert_fullwidth_symbols_ans(text: str) -> str:
    symbol_map = {
        '，': ',',
        '：': ':',
        '（': '(',
        '）': ')',
        '；': ';',
        '“': '"',
        '”': '"',
        '’': "'",
        '‘': "'"
    }
    res = []
    for ch in text:
        res.append(symbol_map.get(ch, ch))
    return "".join(res)

test_snippet = "for i in range（5）：\n    print(i，end=' ')；"
fixed = convert_fullwidth_symbols_ans(test_snippet)

assert "（" not in fixed and "）" not in fixed, "全形括號應被轉換！"
assert "：" not in fixed, "全形冒號應被轉換！"
assert "，" not in fixed, "全形逗號應被轉換！"
assert "；" not in fixed, "全形分號應被轉換！"

# 驗證轉換後的程式碼能否合法編譯
try:
    compile(fixed, "<test>", "exec")
    comp_ok = True
except SyntaxError:
    comp_ok = False

assert comp_ok, "轉換後的代碼應能順利通過編譯！"
print("🎉 13.2.5 所有測試通過！成功掌握等號與標點符號除錯要領！")


### 13.2.6 語法排查實戰演練：1 分鐘極速修正多重語法硬傷程式碼

在真實的 APCS 考場上，時間是無比珍貴的資產。當你在短時間內敲出幾十行程式碼，按下送出卻迎來一面刺眼的黃色 `CE` 時，你絕對不能慌張失措地重寫整份程式，而是要像一位技術精湛的急診外科醫師一樣，以冷靜、迅速的步驟「接單 ➔ 定位 ➔ 止血 ➔ 縫合」！

**考場 1 分鐘語法除錯黃金 SOP**：
1. **第一眼直奔 Traceback 最底部**：不要被中間幾十行的路徑嚇到，直接看最後一行的「錯誤類型」與「行號」。
2. **鎖定報錯行號與指標 `^`**：查看指標指著哪裡，如果是 `expected ':'`，直接跳到該行末尾補冒號。
3. **檢查上一行是否有未閉合括號**：若報錯行看不出任何毛病，立刻往上退一行，檢查 `()`, `[]`, `{}` 是否左右對稱配對。
4. **檢查縮排與全形字元**：若報錯為 `IndentationError` 或 `invalid character`，檢查該行首部是否有空格不一或誤鍵中文全形字元。
5. **本機快速編譯驗證**：在命令列或 Colab 中重新執行一次，確認編譯通過後，才進入真正的邏輯驗證階段。在接下來的實戰大考驗中，我們將挑戰一段集五大語法地雷於一身的災難程式碼！


In [ ]:
# 13.2.6 程式碼演示：實戰除錯操練——診斷集結多重語法硬傷的混合代碼
disaster_code = '''
def check_prime(n)
    if n <= 1:
        return False
    for i in range(2, int(n ** 0.5) + 1:
        if n % i = 0:
            return False
    return True
'''

def diagnose_step_by_step(code_str):
    '''
    逐步排查語法錯誤，直到整份程式碼完全乾淨
    '''
    lines = code_str.strip().splitlines()
    print("=== 原始待診斷程式碼 ===")
    for i, line in enumerate(lines, 1):
        print(f"{i:2d} | {line}")
    print("\n開始直譯器檢驗...")
    
    try:
        compile(code_str, "<disaster>", "exec")
        print("🎉 恭喜！此程式碼結構完美，完全無語法硬傷！")
    except SyntaxError as e:
        print(f"🚨 抓到病灶！第 {e.lineno} 行報錯: {e.msg}")
        print(f"   問題片段: {e.text.strip() if e.text else 'N/A'}")
        print("   處方建議：請檢查冒號、成對括號或等號使用！\n")

diagnose_step_by_step(disaster_code)


### 13.2.6 語法重點回顧與核心觀念提煉

在剛才的災難程式碼 `disaster_code` 中，總共潛伏了三個經典考場地雷：
1. **第 1 行**：`def check_prime(n)` 結尾漏掉冒號 `:`。
2. **第 4 行**：`for i in range(2, int(n ** 0.5) + 1:` 結尾括號不平衡，`range(` 開啟後只有一個右括號，漏掉了閉合外層的右括號 `)`！
3. **第 5 行**：`if n % i = 0:` 在條件判定中誤用了單等號賦值 `=`，應修正為雙等號比較 `==`。

**除錯心理建設**：
直譯器一次「只會回報它遇到的第一個語法錯誤」。當你修好了第 1 行的冒號，按下執行後直譯器又報錯在第 4 行，千萬不要感到沮喪！這代表你的程式碼正在一層層地被修復前進。只要按照「看行號 ➔ 查符號 ➔ 看上行 ➔ 查全形」的節奏，三回合之內必能剷除所有語法硬傷，順利通過編譯期大門！


In [ ]:
# 13.2.6 學生實作練習：限時 1 分鐘外科手術除錯！
# 任務說明：請修復下方的 buggy_solution 程式碼字串，徹底清除所有語法硬傷
# 修正完成後，必須能順利執行 compile 檢查並回傳 True！

buggy_solution = '''
def find_even_sum(numbers)
    total = 0
    for num in numbers:
        if num % 2 = 0:
            total += num
    return total
'''

# 請將完全修復好的程式碼存入 clean_solution 字串中
clean_solution = '''
# 請修正後貼在這邊
'''

# 驗證你的修復成果
diagnose_step_by_step(clean_solution)


In [ ]:
# 13.2.6 單元測試驗證
sample_perfect_solution = '''
def find_even_sum(numbers):
    total = 0
    for num in numbers:
        if num % 2 == 0:
            total += num
    return total
'''

# 語法編譯檢查
compile_success = False
try:
    compile(sample_perfect_solution, "<test>", "exec")
    compile_success = True
except SyntaxError:
    compile_success = False

assert compile_success, "修復後的程式碼必須能通過 compile 編譯！"

# 邏輯執行功能測試
ns = {}
exec(sample_perfect_solution, ns)
find_even_sum = ns['find_even_sum']
assert find_even_sum([1, 2, 3, 4, 5, 6]) == 12, "函式功能應正確計算偶數和！"
assert find_even_sum([1, 3, 5]) == 0, "無偶數時應回傳 0！"

print("🎉 13.2.6 所有測試通過！恭喜你已完全掌握 1 分鐘快速排除 SyntaxError 的外科手術神技！")


## 單元總結與自我評量

### 語法錯誤（SyntaxError）快速排查急救手冊

當你在 APCS 或 Online Judge 上遭遇鮮黃色的 `CE`（Compile Error / SyntaxError）時，請立刻掏出本手冊，依序檢驗以下五大急救項目：

| 排查順序 | 檢查項目 | 典型報錯訊息 | 檢查重點與急救動作 |
| :---: | :--- | :--- | :--- |
| **1** | **冒號檢查** | `expected ':'` | 檢查 `if`, `for`, `while`, `def`, `else`, `elif` 行末是否有英文冒號 `:` |
| **2** | **括號平衡** | `was never closed` 或報錯在下一行 | 檢查 `()`, `[]`, `{}` 是否成對閉合；若本行無誤，**立刻檢查上一行** |
| **3** | **引號完整** | `unterminated string literal` | 檢查字串開頭與結尾是否使用相同引號（`'` 或 `"`），是否有漏打右引號 |
| **4** | **等號辨析** | `Maybe you meant '=='?` | 檢查 `if` / `while` 條件式中是否誤將比較雙等號 `==` 敲成單等號賦值 `=` |
| **5** | **全形標點與縮排** | `invalid character` 或 `IndentationError` | 檢查行首是否混入中文全形空白 `\u3000`，逗號括號是否切回半形輸入 |

---

### 自我實力檢測清單
- [ ] 我能清楚分辨「編譯期語法錯誤（SyntaxError）」與「執行中途崩潰（RuntimeError）」的本質區別。
- [ ] 我知道語法錯誤會導致整份程式碼「連第 1 行都無法執行」，絕不心存僥倖。
- [ ] 我能看懂 Traceback 底部最後一行的錯誤訊息、出錯行號（`lineno`）與游標指標（`^`）。
- [ ] 我牢記「上行漏括號、下行遭殃」的移位陷阱，學會看上一行（Line - 1）排查。
- [ ] 我能在 10 秒內敏銳抓出漏冒號 `:`、單雙等號誤用（`=` vs `==`）與引號未閉合等低級失誤。
- [ ] 我養成使用純英文半形輸入法寫程式碼的衛生習慣，徹底杜絕全形空白 `\u3000` 造成的隱形災難。
